# 18 — Model Portfolio Synthesis

## AI Business Advisor V2

### Objectif

Ce notebook consolide les modèles Data Science validés du projet.

Trois tâches prédictives ont été développées :

1. Claim Occurrence ;
2. Claim Frequency ;
3. Fraud Detection.

L'objectif de cette étape n'est plus d'entraîner ou de tuner
de nouveaux modèles.

Il s'agit de :

- centraliser les performances finales ;
- documenter les entrées et sorties ;
- définir le rôle métier de chaque modèle ;
- documenter les limites ;
- préparer les contrats d'inférence ;
- préparer l'intégration avec l'API et le backend.

Les métriques des trois tâches ne sont pas directement comparables
entre elles car elles répondent à des problèmes prédictifs différents.

In [1]:
import pandas as pd
import numpy as np

import json

from pathlib import Path

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    220
)

In [2]:
cwd = Path.cwd()

possible_data_paths = [
    cwd / "data",
    cwd / "data-science" / "data",
    cwd.parent / "data"
]


DATA_DIR = None

for path in possible_data_paths:

    if path.exists():

        DATA_DIR = path
        break


if DATA_DIR is None:

    raise FileNotFoundError(
        "Impossible de localiser le dossier data"
    )


METADATA_DIR = (
    DATA_DIR
    / "metadata"
)


ARTIFACTS_DIR = (
    DATA_DIR.parent
    / "artifacts"
)


REPORT_DIR = (
    DATA_DIR.parent
    / "reports"
    / "model_portfolio"
)


REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "DATA_DIR :",
    DATA_DIR
)

print(
    "METADATA_DIR :",
    METADATA_DIR
)

print(
    "ARTIFACTS_DIR :",
    ARTIFACTS_DIR
)

print(
    "REPORT_DIR :",
    REPORT_DIR
)

DATA_DIR : c:\Users\user\ai-business-advisor-v2\data-science\data
METADATA_DIR : c:\Users\user\ai-business-advisor-v2\data-science\data\metadata
ARTIFACTS_DIR : c:\Users\user\ai-business-advisor-v2\data-science\artifacts
REPORT_DIR : c:\Users\user\ai-business-advisor-v2\data-science\reports\model_portfolio


In [3]:
with open(
    METADATA_DIR
    / "claim_occurrence_model_v1.json",
    "r",
    encoding="utf-8"
) as f:

    occurrence_metadata = json.load(f)


with open(
    METADATA_DIR
    / "claim_frequency_model_v1.json",
    "r",
    encoding="utf-8"
) as f:

    frequency_metadata = json.load(f)


with open(
    METADATA_DIR
    / "fraud_model_v1.json",
    "r",
    encoding="utf-8"
) as f:

    fraud_metadata = json.load(f)


print(
    "✅ Les trois métadonnées finales sont chargées."
)


✅ Les trois métadonnées finales sont chargées.


In [4]:
artifact_files = {

    "claim_occurrence":
        ARTIFACTS_DIR
        / "claim_occurrence_xgb_isotonic.joblib",

    "claim_frequency":
        ARTIFACTS_DIR
        / "claim_frequency_xgboost_poisson.joblib",

    "fraud":
        ARTIFACTS_DIR
        / "fraud_random_forest_isotonic.joblib"
}


for model_name, path in (
    artifact_files.items()
):

    print(
        model_name,
        "->",
        path.exists(),
        "->",
        path
    )


assert all(
    path.exists()
    for path
    in artifact_files.values()
)


print(
    "\n✅ Tous les artefacts sont disponibles."
)

claim_occurrence -> True -> c:\Users\user\ai-business-advisor-v2\data-science\artifacts\claim_occurrence_xgb_isotonic.joblib
claim_frequency -> True -> c:\Users\user\ai-business-advisor-v2\data-science\artifacts\claim_frequency_xgboost_poisson.joblib
fraud -> True -> c:\Users\user\ai-business-advisor-v2\data-science\artifacts\fraud_random_forest_isotonic.joblib

✅ Tous les artefacts sont disponibles.


In [6]:
model_summary = pd.DataFrame([
    {

        "task":
            "Claim Occurrence",

        "problem_type":
            "Binary Classification",

        "target":
            "ClaimNb > 0",

        "model":
            "XGBoost + Isotonic",

        "primary_metric":
            "PR-AUC",

        "primary_metric_value":
            0.142379,

        "secondary_metric":
            "ROC-AUC",

        "secondary_metric_value":
            0.710567,

        "calibration_metric":
            "Brier",

        "calibration_metric_value":
            0.045621,

        "base_rate":
            0.050252,

        "top_decile_lift":
            3.074,

        "artifact":
            "claim_occurrence_xgb_isotonic.joblib"
    },

    {

        "task":
            "Claim Frequency",

        "problem_type":
            "Count / Frequency Regression",

        "target":
            "ClaimNb / Exposure",

        "model":
            "XGBoost Poisson",

        "primary_metric":
            "Mean Poisson Deviance",

        "primary_metric_value":
            0.301493,

        "secondary_metric":
            "Count MAE",

        "secondary_metric_value":
            0.096294,

        "calibration_metric":
            "Frequency Bias (%)",

        "calibration_metric_value":
            -0.301358,

        "base_rate":
            0.100989,

        "top_decile_lift":
            3.665,

        "artifact":
            "claim_frequency_xgboost_poisson.joblib"
    },

    {

        "task":
            "Fraud Detection",

        "problem_type":
            "Binary Classification",

        "target":
            "FraudFound_P",

        "model":
            "Random Forest + Isotonic",

        "primary_metric":
            "PR-AUC",

        "primary_metric_value":
            0.091127,

        "secondary_metric":
            "ROC-AUC",

        "secondary_metric_value":
            0.674790,

        "calibration_metric":
            "Brier",

        "calibration_metric_value":
            0.051080,

        "base_rate":
            0.052168,

        "top_decile_lift":
            1.832,

        "artifact":
            "fraud_random_forest_isotonic.joblib"
    }
])
model_summary

,task,problem_type,target,model,primary_metric,primary_metric_value,secondary_metric,secondary_metric_value,calibration_metric,calibration_metric_value,base_rate,top_decile_lift,artifact
0,Claim Occurrence,Binary Classification,ClaimNb > 0,XGBoost + Isotonic,PR-AUC,0.142379,ROC-AUC,0.710567,Brier,0.045621,0.050252,3.074,claim_occurrence_xgb_isotonic.joblib
1,Claim Frequency,Count / Frequency Regression,ClaimNb / Exposure,XGBoost Poisson,Mean Poisson Deviance,0.301493,Count MAE,0.096294,Frequency Bias (%),-0.301358,0.100989,3.665,claim_frequency_xgboost_poisson.joblib
2,Fraud Detection,Binary Classification,FraudFound_P,Random Forest + Isotonic,PR-AUC,0.091127,ROC-AUC,0.674790,Brier,0.051080,0.052168,1.832,fraud_random_forest_isotonic.joblib


In [7]:
model_summary.to_csv(
    REPORT_DIR
    / "final_model_portfolio.csv",
    index=False
)

## Interprétation des métriques

Les métriques utilisées dépendent du problème considéré.

### Claim Occurrence

Il s'agit d'une classification binaire fortement déséquilibrée.

La métrique principale est donc la PR-AUC.

### Claim Frequency

Il s'agit d'un problème de comptage avec exposition.

La métrique principale est la Mean Poisson Deviance.

### Fraud Detection

Il s'agit également d'une classification rare et déséquilibrée.

La PR-AUC est utilisée comme métrique principale.

Par conséquent, une valeur de PR-AUC de 0.14 et une Poisson Deviance
de 0.30 ne doivent pas être comparées directement.

Chaque modèle doit être évalué relativement à son propre problème,
à ses baselines et à son protocole expérimental.

In [8]:
business_role = pd.DataFrame([
    {

        "task":
            "Claim Occurrence",

        "business_question":
            "Quelle est la probabilité qu'au moins un sinistre survienne ?",

        "model_output":
            "Probability of claim occurrence",

        "application_role":
            "Risk assessment",

        "automatic_decision":
            False
    },

    {

        "task":
            "Claim Frequency",

        "business_question":
            "Combien de sinistres sont attendus par unité d'exposition ?",

        "model_output":
            "Expected claim frequency",

        "application_role":
            "Portfolio and expected claims analysis",

        "automatic_decision":
            False
    },

    {

        "task":
            "Fraud Detection",

        "business_question":
            "Quels dossiers doivent être priorisés pour investigation ?",

        "model_output":
            "Fraud risk probability / ranking",

        "application_role":
            "Investigation prioritization",

        "automatic_decision":
            False
    }
])


business_role

,task,business_question,model_output,application_role,automatic_decision
0,Claim Occurrence,Quelle est la probabilité qu'au moins un sinis...,Probability of claim occurrence,Risk assessment,False
1,Claim Frequency,Combien de sinistres sont attendus par unité d...,Expected claim frequency,Portfolio and expected claims analysis,False
2,Fraud Detection,Quels dossiers doivent être priorisés pour inv...,Fraud risk probability / ranking,Investigation prioritization,False


In [9]:
business_role.to_csv(
    REPORT_DIR
    / "model_business_roles.csv",
    index=False
)

In [10]:
limitations = pd.DataFrame([
    {

        "task":
            "Claim Occurrence",

        "main_limitations":
            (
                "Rare positive class; technical threshold is not a "
                "business-optimal threshold; SHAP associations are not causal."
            )
    },

    {

        "task":
            "Claim Frequency",

        "main_limitations":
            (
                "No ClaimAmount available; model predicts frequency only; "
                "severity and pure premium cannot be estimated."
            )
    },

    {

        "task":
            "Fraud Detection",

        "main_limitations":
            (
                "Small number of fraud cases; temporal drift; threshold "
                "generalizes less well than ranking; model should prioritize "
                "human investigation rather than automatically reject claims."
            )
    }
])


limitations

,task,main_limitations
0,Claim Occurrence,Rare positive class; technical threshold is no...
1,Claim Frequency,No ClaimAmount available; model predicts frequ...
2,Fraud Detection,Small number of fraud cases; temporal drift; t...


In [11]:
with open(
    METADATA_DIR
    / "feature_manifest.json",
    "r",
    encoding="utf-8"
) as f:

    feature_manifest = json.load(f)

In [12]:
occurrence_config = (
    feature_manifest[
        "claim_occurrence"
    ]
)


occurrence_features = (
    occurrence_config[
        "numeric_features"
    ]
    +
    occurrence_config[
        "categorical_features"
    ]
)


print(
    occurrence_features
)

['Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'log_density', 'Area', 'VehBrand', 'VehGas', 'Region']


In [13]:
frequency_config = (
    feature_manifest[
        "claim_frequency"
    ]
)


frequency_model_features = (
    frequency_config[
        "numeric_features"
    ]
    +
    frequency_config[
        "categorical_features"
    ]
)


print(
    frequency_model_features
)

['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'log_density', 'Area', 'VehBrand', 'VehGas', 'Region']


In [14]:
fraud_config = (
    feature_manifest[
        "fraud"
    ]
)


fraud_features = (
    fraud_config[
        "continuous_features"
    ]
    +
    fraud_config[
        "ordinal_features"
    ]
    +
    fraud_config[
        "categorical_features"
    ]
)


print(
    fraud_features
)

['Age', 'Deductible', 'WeekOfMonth', 'WeekOfMonthClaimed', 'DriverRating', 'Month', 'DayOfWeek', 'Make', 'AccidentArea', 'DayOfWeekClaimed', 'MonthClaimed', 'Sex', 'MaritalStatus', 'VehicleCategory', 'VehiclePrice', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'AgentType', 'NumberOfCars', 'BasePolicy']


In [15]:
serving_contract = {

    "claim_occurrence": {

        "artifact":
            "claim_occurrence_xgb_isotonic.joblib",

        "required_features":
            occurrence_features,

        "outputs": [
            "claim_probability",
            "technical_risk_flag"
        ],

        "meaning":
            (
                "Probability that at least one claim occurs "
                "during the observed exposure period."
            )
    },


    "claim_frequency": {

        "artifact":
            "claim_frequency_xgboost_poisson.joblib",

        "required_model_features":
            frequency_model_features,

        "additional_input": [
            "Exposure"
        ],

        "outputs": [
            "predicted_frequency",
            "expected_claim_count"
        ],

        "formula":
            (
                "expected_claim_count = "
                "predicted_frequency * Exposure"
            )
    },


    "fraud_detection": {

        "artifact":
            "fraud_random_forest_isotonic.joblib",

        "required_features":
            fraud_features,

        "outputs": [
            "fraud_probability",
            "investigation_flag"
        ],

        "threshold":
            0.08585764735167348,

        "meaning":
            (
                "Prioritization score for human fraud investigation. "
                "Not an automatic fraud determination."
            )
    }
}

In [16]:
SERVING_CONTRACT_PATH = (
    METADATA_DIR
    / "model_serving_contract.json"
)


with open(
    SERVING_CONTRACT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        serving_contract,
        f,
        indent=4,
        ensure_ascii=False
    )


print(
    "✅ Serving contract :",
    SERVING_CONTRACT_PATH
)

✅ Serving contract : c:\Users\user\ai-business-advisor-v2\data-science\data\metadata\model_serving_contract.json


# Architecture fonctionnelle Data Science

Les trois modèles répondent à des questions différentes.

## Claim Occurrence

Entrée :

profil du contrat + exposition

Sortie :

probabilité qu'au moins un sinistre survienne.

---

## Claim Frequency

Entrée :

profil du contrat + exposition

Sorties :

- fréquence attendue ;
- nombre attendu de sinistres.

---

## Fraud Detection

Entrée :

informations disponibles sur le dossier de déclaration.

Sorties :

- score/probabilité de fraude ;
- indicateur de priorité d'investigation.

---

## AI Advisor

Le composant LLM ne remplace aucun modèle prédictif.

Il reçoit des sorties structurées produites par les modèles
et peut :

- les synthétiser ;
- les expliquer dans un langage métier ;
- rappeler les limites ;
- produire des recommandations descriptives.

Il ne recalcule pas les scores ML.

# Architecture cible

```text
                     CLIENT / FRONTEND
                            |
                            v
                       SPRING BOOT
                            |
                            v
                 ML INFERENCE SERVICE
                            |
           +----------------+----------------+
           |                |                |
           v                v                v
      OCCURRENCE        FREQUENCY          FRAUD
       MODEL              MODEL            MODEL
           |                |                |
           +----------------+----------------+
                            |
                            v
                 STRUCTURED ML RESULTS
                            |
                 +----------+----------+
                 |                     |
                 v                     v
           BUSINESS RULES          AI ADVISOR
                                  / LLM
                                       |
                                       v
                            NATURAL-LANGUAGE
                               EXPLANATION

In [17]:
report_summary = pd.DataFrame([
    {

        "Module":
            "Claim Occurrence",

        "Final model":
            "XGBoost + Isotonic",

        "Final result":
            "PR-AUC = 0.1424",

        "Business output":
            "Claim probability",

        "Key limitation":
            "Rare event; threshold is technical"
    },

    {

        "Module":
            "Claim Frequency",

        "Final model":
            "XGBoost Poisson",

        "Final result":
            "Poisson Deviance = 0.3015",

        "Business output":
            "Frequency + expected claim count",

        "Key limitation":
            "No severity / ClaimAmount"
    },

    {

        "Module":
            "Fraud Detection",

        "Final model":
            "Random Forest + Isotonic",

        "Final result":
            "PR-AUC = 0.0911",

        "Business output":
            "Investigation priority",

        "Key limitation":
            "Moderate performance and temporal threshold drift"
    }
])


report_summary

,Module,Final model,Final result,Business output,Key limitation
0,Claim Occurrence,XGBoost + Isotonic,PR-AUC = 0.1424,Claim probability,Rare event; threshold is technical
1,Claim Frequency,XGBoost Poisson,Poisson Deviance = 0.3015,Frequency + expected claim count,No severity / ClaimAmount
2,Fraud Detection,Random Forest + Isotonic,PR-AUC = 0.0911,Investigation priority,Moderate performance and temporal threshold drift


In [18]:
report_summary.to_csv(
    REPORT_DIR
    / "report_model_summary.csv",
    index=False
)

In [19]:
generated_files = [
    REPORT_DIR
    / "final_model_portfolio.csv",

    REPORT_DIR
    / "model_business_roles.csv",

    REPORT_DIR
    / "report_model_summary.csv",

    METADATA_DIR
    / "model_serving_contract.json"
]


for path in generated_files:

    print(
        path.name,
        "->",
        path.exists()
    )

final_model_portfolio.csv -> True
model_business_roles.csv -> True
report_model_summary.csv -> True
model_serving_contract.json -> True


In [20]:
print(
    "=" * 90
)

print(
    "AI BUSINESS ADVISOR V2 — FINAL DATA SCIENCE PORTFOLIO"
)

print(
    "=" * 90
)


print(
    "\n=== FINAL MODELS ==="
)

print(
    model_summary
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\n=== BUSINESS ROLES ==="
)

print(
    business_role
    .to_string(
        index=False
    )
)


print(
    "\n=== LIMITATIONS ==="
)

print(
    limitations
    .to_string(
        index=False
    )
)


print(
    "\n=== OCCURRENCE INPUT FEATURES ==="
)

print(
    occurrence_features
)


print(
    "\n=== FREQUENCY MODEL FEATURES ==="
)

print(
    frequency_model_features
)

print(
    "Additional API input : Exposure"
)


print(
    "\n=== FRAUD INPUT FEATURES ==="
)

print(
    fraud_features
)


print(
    "\n=== ARTIFACT CHECK ==="
)

for model_name, path in artifact_files.items():

    print(
        model_name,
        "->",
        path.exists(),
        "->",
        path.name
    )


print(
    "\n=== SERVING CONTRACT ==="
)

print(
    SERVING_CONTRACT_PATH
)


print(
    "\nDATA SCIENCE MODELING PHASE : COMPLETE"
)

print(
    "NEXT PHASE : INFERENCE API + REPORT CONSOLIDATION"
)


print(
    "=" * 90
)

AI BUSINESS ADVISOR V2 — FINAL DATA SCIENCE PORTFOLIO

=== FINAL MODELS ===
            task                 problem_type             target                    model        primary_metric  primary_metric_value secondary_metric  secondary_metric_value calibration_metric  calibration_metric_value  base_rate  top_decile_lift                               artifact
Claim Occurrence        Binary Classification        ClaimNb > 0       XGBoost + Isotonic                PR-AUC              0.142379          ROC-AUC                0.710567              Brier                  0.045621   0.050252            3.074   claim_occurrence_xgb_isotonic.joblib
 Claim Frequency Count / Frequency Regression ClaimNb / Exposure          XGBoost Poisson Mean Poisson Deviance              0.301493        Count MAE                0.096294 Frequency Bias (%)                 -0.301358   0.100989            3.665 claim_frequency_xgboost_poisson.joblib
 Fraud Detection        Binary Classification       FraudFound_

In [21]:
with open(
    METADATA_DIR / "model_serving_contract.json",
    "r",
    encoding="utf-8"
) as f:
    serving_contract = json.load(f)


serving_contract[
    "claim_occurrence"
][
    "threshold"
] = 0.10327080885569255


with open(
    METADATA_DIR / "model_serving_contract.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        serving_contract,
        f,
        indent=4,
        ensure_ascii=False
    )


print(
    serving_contract[
        "claim_occurrence"
    ]
)

{'artifact': 'claim_occurrence_xgb_isotonic.joblib', 'required_features': ['Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'log_density', 'Area', 'VehBrand', 'VehGas', 'Region'], 'outputs': ['claim_probability', 'technical_risk_flag'], 'meaning': 'Probability that at least one claim occurs during the observed exposure period.', 'threshold': 0.10327080885569255}
